In [17]:
knitr::opts_chunk$set(warning=F,message=F,echo=T,include = T)

rm(list=ls())

library(tidyverse)
install.packages("skimr")
library(skimr)

##total for bottom row
sum_rows = function(x) {
  x = as.data.frame(x)
  sums = sapply(x,function(col) if (is.numeric(col)) sum(col, na.rm = T) else NA)
  sums = as.data.frame(t(sums))
  names(sums) = names(x)
  rbind(x, sums)
}

## right column for total
sum_cols = function(x) {
  x$Total = rowSums(x[sapply(x, is.numeric)], na.rm = T)
  x
}

desc_stats = function(x){
  x %>%
    summarise(mean = mean(x,na.rm=T),
              median = median(x,na.rm=T),
              stan_dev = sd(x,na.rm=T))
}

options(scipen=999) # disable scientific notation

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



## Import

In [18]:
data = read.csv("https://drive.google.com/uc?export=download&id=1C0WCLSx2jEsInm_NNbCgNKx6qWRNkp-Q")

names(data) = trimws(str_to_lower(str_replace_all(names(data), " ", "_")))

#sum(duplicated(data$member.id)) no duplicates

skimr::skim(data)

,skim_type,skim_variable,n_missing,complete_rate,character.min,character.max,character.empty,character.n_unique,character.whitespace,logical.mean,logical.count,numeric.mean,numeric.sd,numeric.p0,numeric.p25,numeric.p50,numeric.p75,numeric.p100,numeric.hist
,<chr>,<chr>,<int>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,character,name,0,1,1,30,0,743,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
2,character,location,0,1,4,22,0,294,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
3,character,joined.group.on,0,1,11,18,0,428,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
4,character,last.visited.group.on,0,1,11,18,0,462,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
5,character,last.attended,0,1,0,18,337,26,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
6,character,intro,0,1,2,2,0,1,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
7,character,photo,0,1,2,3,0,2,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
8,character,role,0,1,6,12,0,3,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
9,character,url.of.member.profile,0,1,63,65,0,755,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


## 2) Explore

### 2.1) Count by location

In [19]:
data |>
  group_by(location) |>
  reframe(HEADCOUNT = n()) |>
  arrange(desc(HEADCOUNT)) |>
  head()

data1 = data |>
  mutate(LAST_ATTENDED = mdy(last.attended),
         LAST_VISITED = mdy(last.visited.group.on),
         IN_STATE = case_when(str_detect(location,", SC") ~ "YES",
                           TRUE ~ "NO"))

data2 = data1 |>
  group_by(location) |>
  reframe(RSVP_YES = sum(all.time..yes..rsvps,na.rm = T),
          EVENTS_ATTENDED = sum(events.attended,na.rm = T),
          ATTENDANCE_RATE = round(sum(EVENTS_ATTENDED / RSVP_YES,na.rm = T),2)) |>
  filter(RSVP_YES > 1 & EVENTS_ATTENDED > 1)

location,HEADCOUNT
<chr>,<int>
"Columbia, SC",126
"Charlotte, NC",52
"Lexington, SC",24
"Atlanta, GA",21
"Charleston, SC",18
"Greenville, SC",14


### 2.2) In-state vs out-of-state

In [20]:
data1 |>
  group_by(IN_STATE) |>
  reframe(HEADCOUNT = n()) |>
  sum_rows() |>
  mutate(PERCENT = round(HEADCOUNT/755,2))

IN_STATE,HEADCOUNT,PERCENT
<chr>,<int>,<dbl>
NO,435,0.58
YES,320,0.42
NA,755,1.00


### 2.3) Attendance rate by name and location

In [21]:
data2.1 = data1 |>
  group_by(name,location,IN_STATE,LAST_VISITED,photo) |>
  reframe(RSVP_YES = sum(all.time..yes..rsvps,na.rm = T),
          EVENTS_ATTENDED = sum(events.attended,na.rm = T),
          ATTENDANCE_RATE = round(sum(EVENTS_ATTENDED / RSVP_YES,na.rm = T),2)) |>
  filter(RSVP_YES > 1 & EVENTS_ATTENDED > 1) |>
  arrange(desc(ATTENDANCE_RATE)) |>
  arrange(desc(EVENTS_ATTENDED)) |>
  arrange(desc(LAST_VISITED))

data2.1

name,location,IN_STATE,LAST_VISITED,photo,RSVP_YES,EVENTS_ATTENDED,ATTENDANCE_RATE
<chr>,<chr>,<chr>,<date>,<chr>,<int>,<int>,<dbl>
Alan H. L.,"Sumter, SC",YES,2026-03-09,Yes,2,2,1.00
Carnegie,"Sumter, SC",YES,2026-03-08,Yes,4,2,0.50
Joshua Whitney,"North Charleston, SC",YES,2026-02-25,Yes,3,2,0.67
Jean Yoshii,"Spartanburg, SC",YES,2025-12-30,No,8,7,0.88
Mark Oelkuct,"Aiken, SC",YES,2025-12-29,Yes,30,26,0.87
Tomeka,"Greenville, SC",YES,2025-11-26,Yes,4,3,0.75
Brian Thomas,"Myrtle Beach, SC",YES,2025-11-26,Yes,2,2,1.00
Bret Shroats,"Lexington, KY",NO,2025-07-25,No,2,2,1.00
Garrett Havens,"Columbia, SC",YES,2025-05-29,Yes,6,6,1.00


### 2.4) In-state members only

In [23]:
data2.1 |>
  filter(IN_STATE == 'YES') |>
  select(-IN_STATE)

name,location,LAST_VISITED,photo,RSVP_YES,EVENTS_ATTENDED,ATTENDANCE_RATE
<chr>,<chr>,<date>,<chr>,<int>,<int>,<dbl>
Alan H. L.,"Sumter, SC",2026-03-09,Yes,2,2,1.00
Carnegie,"Sumter, SC",2026-03-08,Yes,4,2,0.50
Joshua Whitney,"North Charleston, SC",2026-02-25,Yes,3,2,0.67
Jean Yoshii,"Spartanburg, SC",2025-12-30,No,8,7,0.88
Mark Oelkuct,"Aiken, SC",2025-12-29,Yes,30,26,0.87
Tomeka,"Greenville, SC",2025-11-26,Yes,4,3,0.75
Brian Thomas,"Myrtle Beach, SC",2025-11-26,Yes,2,2,1.00
Garrett Havens,"Columbia, SC",2025-05-29,Yes,6,6,1.00
Demarithé Wilson,"Lexington, SC",2025-04-03,Yes,2,2,1.00
